In [ ]:
# ALHA: Adaptive Low-Rank Hessian Approximation
Reproduces all experiments from "Adaptive Low-Rank Hessian Approximation for Large-Scale Optimization" (Lagudu, 2026).
Run all cells in order to reproduce results.

In [11]:
import os
os.makedirs('alha/src', exist_ok=True)
os.makedirs('alha/experiments', exist_ok=True)
os.makedirs('alha/results', exist_ok=True)

In [13]:
%%writefile alha/src/alha.py
import numpy as np
from dataclasses import dataclass, field
from typing import Callable, Optional
import time


@dataclass
class ALHAConfig:
    m_min: int = 2
    m_max: int = 50
    m_0: int = 5
    eps_low: float = 0.1
    eps_high: float = 0.3
    delta_m_plus: int = 2
    delta_m_minus: int = 1
    c1: float = 1e-4
    c2: float = 0.9
    eps_curv: float = 1e-8
    gamma_min: float = 1e-8
    gamma_max: float = 1e8
    q_clip_low: float = 0.01
    q_clip_high: float = 100.0
    eps_tol: float = 1e-8
    max_iter: int = 10000
    verbose: bool = False
    log_interval: int = 100


@dataclass
class ALHAResult:
    x: np.ndarray
    f_val: float
    grad_norm: float
    n_iter: int
    converged: bool
    time_elapsed: float
    f_history: list = field(default_factory=list)
    grad_norm_history: list = field(default_factory=list)
    rank_history: list = field(default_factory=list)
    q_history: list = field(default_factory=list)


def two_loop_recursion(g, S, Y, m, gamma=1.0):
    ell = min(len(S), m)
    if ell == 0:
        return gamma * g

    q = g.copy()
    alphas = np.zeros(ell)
    rhos = np.zeros(ell)

    for i in range(ell - 1, -1, -1):
        sy = float(np.dot(S[i], Y[i]))
        if abs(sy) < 1e-15:
            continue
        rhos[i] = 1.0 / sy
        alphas[i] = rhos[i] * float(np.dot(S[i], q))
        q = q - alphas[i] * Y[i]

    r = gamma * q

    for i in range(ell):
        if rhos[i] == 0.0:
            continue
        beta_i = rhos[i] * float(np.dot(Y[i], r))
        r = r + (alphas[i] - beta_i) * S[i]

    return r


def compute_quality(S, Y, gamma, clip_low=0.01, clip_high=100.0):
    """
    Compute approximation quality using HELD-OUT pairs.
    
    BUG FIX: Previously passed the current (s,y) pair which had just been
    added to S,Y — measuring quality on training data, always returning ~1.0.
    
    Fix: Use the oldest stored pair as a held-out validation pair.
    Compute H * y_old using all OTHER pairs, then compare to s_old.
    This gives a true out-of-sample quality estimate.
    """
    if len(S) < 2:
        # Not enough pairs to do held-out evaluation — assume neutral quality
        return 0.5

    # Hold out the oldest pair for validation
    s_val = S[0]
    y_val = Y[0]
    S_train = S[1:]
    Y_train = Y[1:]

    sy_val = float(np.dot(s_val, y_val))
    if abs(sy_val) < 1e-15:
        return 1.0

    # Apply H (built from training pairs) to y_val
    h = two_loop_recursion(y_val, S_train, Y_train, len(S_train), gamma)
    hy = float(np.dot(h, y_val))

    q = float(np.clip(hy / sy_val, clip_low, clip_high))
    return q


def _strong_wolfe(f, grad_f, x, d, f0, g0, c1=1e-4, c2=0.9, alpha_max=50.0):
    """Strong Wolfe line search."""
    derphi0 = float(np.dot(g0, d))
    if derphi0 >= 0:
        return 1e-4

    alpha_prev = 0.0
    alpha = min(1.0, alpha_max)
    f_prev = f0
    max_ls = 40

    for i in range(max_ls):
        x_new = x + alpha * d
        f_new = float(f(x_new))

        if f_new > f0 + c1 * alpha * derphi0 or (i > 0 and f_new >= f_prev):
            return _zoom(f, grad_f, x, d, f0, derphi0, alpha_prev, alpha,
                         f_prev, f_new, c1, c2)

        g_new = grad_f(x_new)
        derphi_new = float(np.dot(g_new, d))

        if abs(derphi_new) <= c2 * abs(derphi0):
            return alpha

        if derphi_new >= 0:
            return _zoom(f, grad_f, x, d, f0, derphi0, alpha, alpha_prev,
                         f_new, f_prev, c1, c2)

        alpha_prev = alpha
        f_prev = f_new
        alpha = min(alpha * 2.0, alpha_max)

    return alpha


def _zoom(f, grad_f, x, d, f0, derphi0, a_lo, a_hi, f_lo, f_hi, c1, c2):
    for _ in range(30):
        alpha = 0.5 * (a_lo + a_hi)
        x_new = x + alpha * d
        f_new = float(f(x_new))

        if f_new > f0 + c1 * alpha * derphi0 or f_new >= f_lo:
            a_hi = alpha
            f_hi = f_new
        else:
            g_new = grad_f(x_new)
            derphi_new = float(np.dot(g_new, d))

            if abs(derphi_new) <= c2 * abs(derphi0):
                return alpha

            if derphi_new * (a_hi - a_lo) >= 0:
                a_hi = a_lo
                f_hi = f_lo

            a_lo = alpha
            f_lo = f_new

        if abs(a_hi - a_lo) < 1e-14:
            break

    return 0.5 * (a_lo + a_hi)


def alha(f, grad_f, x0, config=None):
    if config is None:
        config = ALHAConfig()

    x = x0.copy().astype(np.float64)
    S, Y = [], []
    g = grad_f(x)
    m_k = config.m_0
    gamma = 1.0

    f_history = [float(f(x))]
    grad_norm_history = [float(np.linalg.norm(g))]
    rank_history = [m_k]
    q_history = [1.0]

    t_start = time.time()
    converged = False
    n_iter = 0

    for k in range(config.max_iter):
        grad_norm = float(np.linalg.norm(g))

        if config.verbose and k % config.log_interval == 0:
            print(f"  Iter {k:5d} | f={f_history[-1]:.4e} | ||g||={grad_norm:.3e} | m={m_k}")

        if grad_norm < config.eps_tol:
            converged = True
            n_iter = k
            break

        # Search direction
        d_k = -two_loop_recursion(g, S, Y, m_k, gamma)

        # Ensure descent
        if float(np.dot(g, d_k)) >= 0:
            d_k = -g.copy()

        f_val = float(f(x))

        # Strong Wolfe line search
        alpha_k = _strong_wolfe(f, grad_f, x, d_k, f_val, g,
                                c1=config.c1, c2=config.c2)

        if alpha_k is None or alpha_k < 1e-14:
            alpha_k = 1e-4

        # Step
        x_new = x + alpha_k * d_k
        g_new = grad_f(x_new)

        s_k = x_new - x
        y_k = g_new - g
        sy = float(np.dot(s_k, y_k))
        curv_thr = config.eps_curv * float(np.linalg.norm(s_k)) * float(np.linalg.norm(y_k))

        # Add pair BEFORE quality check (so we have at least 2 pairs for held-out eval)
        if sy > curv_thr:
            S.append(s_k.copy())
            Y.append(y_k.copy())
            yy = float(np.dot(y_k, y_k))
            if yy > 1e-15:
                gamma = float(np.clip(sy / yy, config.gamma_min, config.gamma_max))
            while len(S) > m_k:
                S.pop(0)
                Y.pop(0)

        # Quality metric and rank adaptation
        # FIX: compute_quality now uses held-out oldest pair, not current pair
        q_hat = compute_quality(S, Y, gamma,
                                config.q_clip_low, config.q_clip_high)

        if q_hat < 1.0 - config.eps_high:
            m_k = min(m_k + config.delta_m_plus, config.m_max)
        elif q_hat > 1.0 - config.eps_low:
            m_k = max(m_k - config.delta_m_minus, config.m_min)

        x = x_new
        g = g_new
        n_iter = k + 1

        f_history.append(float(f(x)))
        grad_norm_history.append(float(np.linalg.norm(g)))
        rank_history.append(m_k)
        q_history.append(q_hat)

    t_elapsed = time.time() - t_start

    return ALHAResult(
        x=x,
        f_val=float(f(x)),
        grad_norm=float(np.linalg.norm(g)),
        n_iter=n_iter,
        converged=converged,
        time_elapsed=t_elapsed,
        f_history=f_history,
        grad_norm_history=grad_norm_history,
        rank_history=rank_history,
        q_history=q_history,
    )
    

Overwriting alha/src/alha.py


In [17]:
%%writefile alha/src/problems.py

"""
Test problems used in the ALHA paper experiments.

Problems:
    1. Well-conditioned quadratic  (kappa = 10)
    2. Ill-conditioned quadratic   (kappa = 1e4)
    3. Rosenbrock function
    4. Logistic regression (MNIST)
    5. Neural network training (MNIST)
    6. Sparse logistic regression  (RCV1)
"""

import numpy as np
from dataclasses import dataclass
from typing import Tuple, Callable


@dataclass
class Problem:
    """Container for an optimization problem."""
    name: str
    f: Callable
    grad_f: Callable
    x0: np.ndarray
    d: int
    description: str = ""


# ---------------------------------------------------------------------------
# Problem 1 & 2: Quadratic f(x) = 0.5 * x^T A x
# ---------------------------------------------------------------------------

def make_quadratic(d: int = 1000, kappa: float = 10.0, seed: int = 42) -> Problem:
    rng = np.random.RandomState(seed)
    Q, _ = np.linalg.qr(rng.randn(d, d))
    lambdas = np.linspace(1.0, kappa, d)
    A = Q @ np.diag(lambdas) @ Q.T
    A = 0.5 * (A + A.T)
    x0 = rng.randn(d)

    def f(x):
        return 0.5 * float(x @ A @ x)

    def grad_f(x):
        return A @ x

    name = f"Quadratic (kappa={kappa:.0e}, d={d})"
    return Problem(name=name, f=f, grad_f=grad_f, x0=x0, d=d,
                   description=f"Quadratic with condition number {kappa}")


# ---------------------------------------------------------------------------
# Problem 3: Rosenbrock function
# ---------------------------------------------------------------------------

def make_rosenbrock(d: int = 100, seed: int = 42) -> Problem:
    rng = np.random.RandomState(seed)
    x0 = rng.randn(d) * 0.5

    def f(x):
        return float(np.sum(
            100.0 * (x[1:] - x[:-1] ** 2) ** 2 + (1.0 - x[:-1]) ** 2
        ))

    def grad_f(x):
        g = np.zeros_like(x)
        g[:-1] += -400.0 * x[:-1] * (x[1:] - x[:-1] ** 2) - 2.0 * (1.0 - x[:-1])
        g[1:] += 200.0 * (x[1:] - x[:-1] ** 2)
        return g

    return Problem(
        name=f"Rosenbrock (d={d})",
        f=f, grad_f=grad_f, x0=x0, d=d,
        description="Extended Rosenbrock function"
    )


# ---------------------------------------------------------------------------
# Problem 4: Logistic Regression on real MNIST (binary: 0 vs 1)
# ---------------------------------------------------------------------------

def make_logistic_regression(
    n: int = 12665,
    d: int = 784,
    lam: float = 1e-4,
    seed: int = 42,
    use_real_mnist: bool = True
) -> Problem:
    """
    Binary logistic regression with L2 regularization.
    Uses real MNIST (digits 0 vs 1) if available, else synthetic data.
    """
    rng = np.random.RandomState(seed)

    if use_real_mnist:
        try:
            from sklearn.datasets import fetch_openml
            print("  Loading MNIST for logistic regression...")
            mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
            X_all = mnist.data.astype(np.float64) / 255.0
            y_all = mnist.target.astype(int)
            # Binary: digits 0 vs 1
            mask = (y_all == 0) | (y_all == 1)
            X_all = X_all[mask]
            y_all = y_all[mask]
            y_all = np.where(y_all == 0, -1, 1)
            # Use up to n samples
            idx = rng.permutation(len(X_all))[:n]
            X = X_all[idx]
            y = y_all[idx]
            n_actual = len(X)
            print(f"  Loaded real MNIST binary: n={n_actual}, d={d}")
        except Exception as e:
            print(f"  MNIST load failed ({e}), using synthetic data")
            use_real_mnist = False

    if not use_real_mnist:
        X = rng.randn(n, d) / np.sqrt(d)
        y = rng.choice([-1, 1], size=n)

    w0 = np.zeros(d)
    n_actual = len(X)

    def f(w):
        margins = y * (X @ w)
        loss = np.mean(np.log1p(np.exp(-np.clip(margins, -500, 500))))
        reg = 0.5 * lam * np.dot(w, w)
        return float(loss + reg)

    def grad_f(w):
        margins = y * (X @ w)
        sigmoid = 1.0 / (1.0 + np.exp(np.clip(margins, -500, 500)))
        g = -(X.T @ (y * sigmoid)) / n_actual + lam * w
        return g

    return Problem(
        name=f"Logistic Regression MNIST (n={n_actual}, d={d})",
        f=f, grad_f=grad_f, x0=w0, d=d,
        description="Binary logistic regression on MNIST (0 vs 1)"
    )


# ---------------------------------------------------------------------------
# Problem 5: Two-layer neural network on real MNIST (10-class)
# ---------------------------------------------------------------------------

def make_neural_network(
    n: int = 5000,
    d_in: int = 784,
    d_hidden: int = 50,
    d_out: int = 10,
    lam: float = 1e-4,
    seed: int = 42,
    use_real_mnist: bool = True
) -> Problem:
    """
    Two-layer MLP on MNIST (10-class).

    KEY FIXES vs previous version:
    - Uses real MNIST data (not random) so the problem is actually learnable
    - Reduced d_hidden=50 (vs 100) to keep d_total manageable (~40k params)
    - n=5000 samples >> d_total so system is overdetermined
    - He initialization for better conditioning

    Architecture: Linear(d_in, d_hidden) -> ReLU -> Linear(d_hidden, d_out)
    Total params: 784*50 + 50 + 50*10 + 10 = 39760
    """
    rng = np.random.RandomState(seed)

    if use_real_mnist:
        try:
            from sklearn.datasets import fetch_openml
            print("  Loading MNIST for neural network...")
            mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
            X_all = mnist.data.astype(np.float64) / 255.0
            y_all = mnist.target.astype(int)
            idx = rng.permutation(len(X_all))[:n]
            X = X_all[idx]
            labels = y_all[idx]
            print(f"  Loaded real MNIST 10-class: n={n}, d={d_in}")
        except Exception as e:
            print(f"  MNIST load failed ({e}), using synthetic data")
            use_real_mnist = False

    if not use_real_mnist:
        X = rng.randn(n, d_in) / np.sqrt(d_in)
        labels = rng.randint(0, d_out, size=n)

    Y = np.eye(d_out)[labels]  # one-hot (n, 10)

    # Parameter layout
    n_W1 = d_in * d_hidden
    n_b1 = d_hidden
    n_W2 = d_hidden * d_out
    n_b2 = d_out
    d_total = n_W1 + n_b1 + n_W2 + n_b2

    # He initialization — much better conditioned than 0.01 * randn
    theta0 = np.concatenate([
        rng.randn(n_W1) * np.sqrt(2.0 / d_in),   # W1
        np.zeros(n_b1),                             # b1
        rng.randn(n_W2) * np.sqrt(2.0 / d_hidden), # W2
        np.zeros(n_b2),                             # b2
    ])

    def unpack(theta):
        W1 = theta[:n_W1].reshape(d_in, d_hidden)
        b1 = theta[n_W1:n_W1 + n_b1]
        W2 = theta[n_W1 + n_b1:n_W1 + n_b1 + n_W2].reshape(d_hidden, d_out)
        b2 = theta[n_W1 + n_b1 + n_W2:]
        return W1, b1, W2, b2

    def forward(theta):
        W1, b1, W2, b2 = unpack(theta)
        Z1 = X @ W1 + b1
        A1 = np.maximum(0, Z1)      # ReLU
        Z2 = A1 @ W2 + b2
        Z2 -= Z2.max(axis=1, keepdims=True)  # numerical stability
        exp_Z2 = np.exp(Z2)
        probs = exp_Z2 / exp_Z2.sum(axis=1, keepdims=True)
        return Z1, A1, probs

    def f(theta):
        _, _, probs = forward(theta)
        log_probs = np.log(np.clip(probs, 1e-15, 1.0))
        loss = -np.mean(np.sum(Y * log_probs, axis=1))
        reg = 0.5 * lam * np.dot(theta, theta)
        return float(loss + reg)

    def grad_f(theta):
        W1, b1, W2, b2 = unpack(theta)
        Z1, A1, probs = forward(theta)
        delta2 = (probs - Y) / n
        dW2 = A1.T @ delta2
        db2 = delta2.sum(axis=0)
        delta1 = (delta2 @ W2.T) * (Z1 > 0)
        dW1 = X.T @ delta1
        db1 = delta1.sum(axis=0)
        g = np.concatenate([dW1.ravel(), db1, dW2.ravel(), db2]) + lam * theta
        return g

    def accuracy(theta):
        _, _, probs = forward(theta)
        return float(np.mean(probs.argmax(axis=1) == labels))

    # Attach accuracy for reporting
    problem = Problem(
        name=f"Neural Network MNIST (n={n}, d={d_total})",
        f=f, grad_f=grad_f, x0=theta0, d=d_total,
        description="Two-layer MLP with ReLU on real MNIST"
    )
    problem.accuracy = accuracy  # attach for use in run_experiments
    return problem


# ---------------------------------------------------------------------------
# Problem 6: Sparse logistic regression with smooth L1
# ---------------------------------------------------------------------------

def make_sparse_logistic(
    n: int = 2000,
    d: int = 5000,
    lam: float = 1e-3,
    mu_smooth: float = 1e-3,
    seed: int = 42
) -> Problem:
    rng = np.random.RandomState(seed)
    nnz_per_row = max(1, d // 20)
    X = np.zeros((n, d))
    for i in range(n):
        idx = rng.choice(d, nnz_per_row, replace=False)
        X[i, idx] = rng.randn(nnz_per_row)
    X /= np.sqrt(nnz_per_row)
    y = rng.choice([-1, 1], size=n)
    w0 = np.zeros(d)

    def smooth_l1(w):
        return np.sum(np.sqrt(w ** 2 + mu_smooth ** 2) - mu_smooth)

    def grad_smooth_l1(w):
        return w / np.sqrt(w ** 2 + mu_smooth ** 2)

    def f(w):
        margins = y * (X @ w)
        loss = np.mean(np.log1p(np.exp(-np.clip(margins, -500, 500))))
        reg = lam * smooth_l1(w)
        return float(loss + reg)

    def grad_f(w):
        margins = y * (X @ w)
        sigmoid = 1.0 / (1.0 + np.exp(np.clip(margins, -500, 500)))
        g = -(X.T @ (y * sigmoid)) / n + lam * grad_smooth_l1(w)
        return g

    return Problem(
        name=f"Sparse Logistic (n={n}, d={d})",
        f=f, grad_f=grad_f, x0=w0, d=d,
        description="Sparse logistic regression with smooth L1 regularization"
    )


def get_all_problems() -> dict:
    """Return all six test problems from the paper."""
    return {
        "well_conditioned_quadratic": make_quadratic(d=1000, kappa=10.0),
        "ill_conditioned_quadratic":  make_quadratic(d=1000, kappa=1e4),
        "rosenbrock":                 make_rosenbrock(d=100),
        "logistic_mnist":             make_logistic_regression(
                                          n=12665, d=784, use_real_mnist=True),
        "neural_network":             make_neural_network(
                                          n=5000, d_hidden=50, use_real_mnist=True),
        "sparse_logistic":            make_sparse_logistic(n=2000, d=5000),
    }


Overwriting alha/src/problems.py


In [19]:
%%writefile alha/src/baselines.py
"""
Baseline optimizers for comparison with ALHA.
Includes: Gradient Descent, L-BFGS, Adam.
"""

import numpy as np
from scipy.optimize import minimize
from dataclasses import dataclass, field
from typing import Callable, Optional
import time


@dataclass
class BaselineResult:
    """Result object for baseline optimizers."""
    x: np.ndarray
    f_val: float
    grad_norm: float
    n_iter: int
    converged: bool
    time_elapsed: float
    f_history: list = field(default_factory=list)
    grad_norm_history: list = field(default_factory=list)


def run_gradient_descent(
    f: Callable,
    grad_f: Callable,
    x0: np.ndarray,
    lr: float = 0.01,
    max_iter: int = 10000,
    eps_tol: float = 1e-8,
    verbose: bool = False
) -> BaselineResult:
    """Gradient descent with fixed step size."""
    x = x0.copy().astype(float)
    f_history = [f(x)]
    grad_norm_history = [np.linalg.norm(grad_f(x))]

    t_start = time.time()
    converged = False
    n_iter = 0

    for k in range(max_iter):
        g = grad_f(x)
        grad_norm = np.linalg.norm(g)

        if grad_norm < eps_tol:
            converged = True
            n_iter = k
            break

        x = x - lr * g
        n_iter = k + 1
        f_history.append(f(x))
        grad_norm_history.append(np.linalg.norm(grad_f(x)))

        if verbose and k % 500 == 0:
            print(f"  GD Iter {k:5d} | f={f(x):.6e} | ||g||={grad_norm:.3e}")

    t_elapsed = time.time() - t_start
    g_final = grad_f(x)

    return BaselineResult(
        x=x, f_val=f(x), grad_norm=np.linalg.norm(g_final),
        n_iter=n_iter, converged=converged, time_elapsed=t_elapsed,
        f_history=f_history, grad_norm_history=grad_norm_history
    )


def run_lbfgs(
    f: Callable,
    grad_f: Callable,
    x0: np.ndarray,
    m: int = 10,
    max_iter: int = 10000,
    eps_tol: float = 1e-8,
    verbose: bool = False
) -> BaselineResult:
    """
    L-BFGS with fixed memory parameter m.
    Uses scipy's implementation wrapped to collect history.
    """
    x = x0.copy().astype(float)
    f_history = [f(x)]
    grad_norm_history = [np.linalg.norm(grad_f(x))]
    iter_count = [0]

    def callback(xk):
        iter_count[0] += 1
        fval = f(xk)
        gnorm = np.linalg.norm(grad_f(xk))
        f_history.append(fval)
        grad_norm_history.append(gnorm)
        if verbose and iter_count[0] % 100 == 0:
            print(f"  L-BFGS(m={m}) Iter {iter_count[0]:5d} | f={fval:.6e} | ||g||={gnorm:.3e}")

    t_start = time.time()

    result = minimize(
        fun=f,
        x0=x,
        jac=grad_f,
        method='L-BFGS-B',
        callback=callback,
        options={
            'maxiter': max_iter,
            'ftol': 0,
            'gtol': eps_tol,
            'maxcor': m,
            'maxfun': max_iter * 20,
        }
    )

    t_elapsed = time.time() - t_start

    return BaselineResult(
        x=result.x,
        f_val=float(result.fun),
        grad_norm=np.linalg.norm(grad_f(result.x)),
        n_iter=iter_count[0],
        converged=result.success,
        time_elapsed=t_elapsed,
        f_history=f_history,
        grad_norm_history=grad_norm_history
    )


def run_adam(
    f: Callable,
    grad_f: Callable,
    x0: np.ndarray,
    lr: float = 1e-3,
    beta1: float = 0.9,
    beta2: float = 0.999,
    eps: float = 1e-8,
    max_iter: int = 10000,
    eps_tol: float = 1e-8,
    verbose: bool = False
) -> BaselineResult:
    """Adam optimizer."""
    x = x0.copy().astype(float)
    m_adam = np.zeros_like(x)
    v_adam = np.zeros_like(x)
    f_history = [f(x)]
    grad_norm_history = [np.linalg.norm(grad_f(x))]

    t_start = time.time()
    converged = False
    n_iter = 0

    for k in range(1, max_iter + 1):
        g = grad_f(x)
        grad_norm = np.linalg.norm(g)

        if grad_norm < eps_tol:
            converged = True
            n_iter = k - 1
            break

        m_adam = beta1 * m_adam + (1 - beta1) * g
        v_adam = beta2 * v_adam + (1 - beta2) * g ** 2

        m_hat = m_adam / (1 - beta1 ** k)
        v_hat = v_adam / (1 - beta2 ** k)

        x = x - lr * m_hat / (np.sqrt(v_hat) + eps)
        n_iter = k
        f_history.append(f(x))
        grad_norm_history.append(np.linalg.norm(grad_f(x)))

        if verbose and k % 500 == 0:
            print(f"  Adam Iter {k:5d} | f={f(x):.6e} | ||g||={grad_norm:.3e}")

    t_elapsed = time.time() - t_start

    return BaselineResult(
        x=x, f_val=f(x), grad_norm=np.linalg.norm(grad_f(x)),
        n_iter=n_iter, converged=converged, time_elapsed=t_elapsed,
        f_history=f_history, grad_norm_history=grad_norm_history
    )


Overwriting alha/src/baselines.py


In [21]:
%%writefile alha/experiments/run_experiments.py

"""
run_experiments.py — ALHA paper experiments
"""

import sys
import os
import argparse
import json
import time
import numpy as np

# Ensure src is found before the alha package directory
sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'src'))

# Force reimport from src, not package directory
import importlib
for key in list(sys.modules.keys()):
    if key in ('alha', 'problems', 'baselines'):
        del sys.modules[key]

from alha import alha, ALHAConfig
from baselines import run_gradient_descent, run_lbfgs, run_adam
from problems import get_all_problems


def run_single_problem(name: str, problem, verbose: bool = False) -> dict:
    print(f"\n{'='*60}")
    print(f"Problem: {problem.name}")
    print(f"d = {problem.d}")
    print(f"{'='*60}")

    results = {}
    has_accuracy = hasattr(problem, 'accuracy')

    def get_acc(x):
        if has_accuracy:
            return round(float(problem.accuracy(x)) * 100, 2)
        return None

    # --- ALHA ---
    print("\nRunning ALHA...")
    config = ALHAConfig(verbose=verbose, eps_tol=1e-8, max_iter=10000)
    result = alha(problem.f, problem.grad_f, problem.x0.copy(), config)
    acc = get_acc(result.x)
    results['ALHA'] = {
        'iterations': result.n_iter,
        'time': round(result.time_elapsed, 3),
        'final_grad_norm': float(result.grad_norm),
        'final_f': float(result.f_val),
        'converged': result.converged,
        'avg_rank': round(float(np.mean(result.rank_history)), 1),
        'accuracy': acc,
        'f_history': [float(v) for v in result.f_history],
        'grad_norm_history': [float(v) for v in result.grad_norm_history],
        'rank_history': result.rank_history,
    }
    acc_str = f" | Acc: {acc:.2f}%" if acc is not None else ""
    print(f"  Iterations: {result.n_iter} | Time: {result.time_elapsed:.3f}s | "
          f"||g||: {result.grad_norm:.2e} | Avg rank: {np.mean(result.rank_history):.1f}{acc_str}")

    # --- L-BFGS variants ---
    for m in [5, 10, 20]:
        print(f"\nRunning L-BFGS (m={m})...")
        result = run_lbfgs(problem.f, problem.grad_f, problem.x0.copy(),
                           m=m, max_iter=10000, eps_tol=1e-8, verbose=verbose)
        acc = get_acc(result.x)
        key = f'L-BFGS (m={m})'
        results[key] = {
            'iterations': result.n_iter,
            'time': round(result.time_elapsed, 3),
            'final_grad_norm': float(result.grad_norm),
            'final_f': float(result.f_val),
            'converged': result.converged,
            'avg_rank': m,
            'accuracy': acc,
            'f_history': [float(v) for v in result.f_history],
            'grad_norm_history': [float(v) for v in result.grad_norm_history],
        }
        acc_str = f" | Acc: {acc:.2f}%" if acc is not None else ""
        print(f"  Iterations: {result.n_iter} | Time: {result.time_elapsed:.3f}s | "
              f"||g||: {result.grad_norm:.2e}{acc_str}")

    # --- Adam ---
    print("\nRunning Adam...")
    result = run_adam(problem.f, problem.grad_f, problem.x0.copy(),
                      lr=1e-3, max_iter=10000, eps_tol=1e-8, verbose=verbose)
    acc = get_acc(result.x)
    results['Adam'] = {
        'iterations': result.n_iter,
        'time': round(result.time_elapsed, 3),
        'final_grad_norm': float(result.grad_norm),
        'final_f': float(result.f_val),
        'converged': result.converged,
        'avg_rank': None,
        'accuracy': acc,
        'f_history': [float(v) for v in result.f_history],
        'grad_norm_history': [float(v) for v in result.grad_norm_history],
    }
    acc_str = f" | Acc: {acc:.2f}%" if acc is not None else ""
    print(f"  Iterations: {result.n_iter} | Time: {result.time_elapsed:.3f}s | "
          f"||g||: {result.grad_norm:.2e}{acc_str}")

    # --- Gradient Descent ---
    print("\nRunning Gradient Descent...")
    g0 = problem.grad_f(problem.x0)
    lr = min(0.01, 1.0 / (np.linalg.norm(g0) + 1e-8))
    result = run_gradient_descent(problem.f, problem.grad_f, problem.x0.copy(),
                                  lr=lr, max_iter=10000, eps_tol=1e-8, verbose=verbose)
    acc = get_acc(result.x)
    results['GD'] = {
        'iterations': result.n_iter,
        'time': round(result.time_elapsed, 3),
        'final_grad_norm': float(result.grad_norm),
        'final_f': float(result.f_val),
        'converged': result.converged,
        'avg_rank': None,
        'accuracy': acc,
        'f_history': [float(v) for v in result.f_history],
        'grad_norm_history': [float(v) for v in result.grad_norm_history],
    }
    acc_str = f" | Acc: {acc:.2f}%" if acc is not None else ""
    print(f"  Iterations: {result.n_iter} | Time: {result.time_elapsed:.3f}s | "
          f"||g||: {result.grad_norm:.2e}{acc_str}")

    return results


def print_summary_table(name: str, results: dict):
    has_acc = any(r.get('accuracy') is not None for r in results.values())
    print(f"\n--- Summary: {name} ---")
    if has_acc:
        print(f"{'Method':<20} {'Iters':>8} {'Time(s)':>10} {'||grad||':>12} {'Avg Rank':>10} {'Acc%':>8}")
        print("-" * 75)
    else:
        print(f"{'Method':<20} {'Iters':>8} {'Time(s)':>10} {'||grad||':>12} {'Avg Rank':>10}")
        print("-" * 65)

    for method, r in results.items():
        rank_str = f"{r['avg_rank']:.1f}" if r['avg_rank'] is not None else "--"
        converged_str = "" if r['converged'] else " (NC)"
        acc_str = f"{r['accuracy']:.2f}" if r.get('accuracy') is not None else "--"
        if has_acc:
            print(f"{method:<20} {r['iterations']:>8} {r['time']:>10.3f} "
                  f"{r['final_grad_norm']:>12.3e} {rank_str:>10} {acc_str:>8}{converged_str}")
        else:
            print(f"{method:<20} {r['iterations']:>8} {r['time']:>10.3f} "
                  f"{r['final_grad_norm']:>12.3e} {rank_str:>10}{converged_str}")


def main():
    parser = argparse.ArgumentParser(description='Run ALHA experiments')
    parser.add_argument('--problem', type=str, default='all')
    parser.add_argument('--verbose', action='store_true')
    parser.add_argument('--output_dir', type=str, default='results')
    args = parser.parse_args()

    os.makedirs(args.output_dir, exist_ok=True)
    all_problems = get_all_problems()

    if args.problem == 'all':
        problems_to_run = all_problems
    else:
        if args.problem not in all_problems:
            print(f"Unknown problem: {args.problem}")
            print(f"Available: {list(all_problems.keys())}")
            sys.exit(1)
        problems_to_run = {args.problem: all_problems[args.problem]}

    all_results = {}
    t_total_start = time.time()

    for name, problem in problems_to_run.items():
        results = run_single_problem(name, problem, verbose=args.verbose)
        all_results[name] = results
        print_summary_table(name, results)

        out_path = os.path.join(args.output_dir, f'{name}.json')
        with open(out_path, 'w') as fp:
            json.dump(results, fp, indent=2)
        print(f"\nSaved results to {out_path}")

    combined_path = os.path.join(args.output_dir, 'all_results.json')
    with open(combined_path, 'w') as fp:
        json.dump(all_results, fp, indent=2)

    t_total = time.time() - t_total_start
    print(f"\n{'='*60}")
    print(f"All experiments complete. Total time: {t_total:.1f}s")
    print(f"Results saved to: {args.output_dir}/")
    print(f"{'='*60}")


if __name__ == '__main__':
    main()

Overwriting alha/experiments/run_experiments.py


In [ ]:
import subprocess
result = subprocess.run(
    ['python', 'alha/experiments/run_experiments.py', '--problem', 'all'],
    capture_output=True, text=True, cwd='/kaggle/working'
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
import json
with open('alha/results/all_results.json') as f:
    data = json.load(f)
for problem, methods in data.items():
    print(f"\n=== {problem} ===")
    for method, r in methods.items():
        print(f"  {method}: iters={r['iterations']}, time={r['time']}s, grad={r['final_grad_norm']:.2e}")